In [ ]:
import pandas as pd
import os

INPUT  = r"C:\Users\asus\Desktop\New_Dataset\AQI_Gaushala_Kathmandu.csv"
OUTPUT = r"C:\Users\asus\Desktop\New_Dataset\AQI_Gaushala_Kathmandu_Cleaned.csv"

df = pd.read_csv(INPUT)

print("=" * 55)
print("  CLEANING: Gaushala Kathmandu AQI")
print("=" * 55)
print(f"  Raw rows     : {len(df):,}")
print(f"  Parameters   : {df['parameter'].unique().tolist()}")

df["datetime_utc"] = pd.to_datetime(df["datetime_utc"], utc=True, errors="coerce")
df["datetime_local"] = pd.to_datetime(df["datetime_local"], errors="coerce")

df = df.dropna(subset=["value", "datetime_utc"])
print(f"  After null drop  : {len(df):,} rows")

before = len(df)
df = df[~((df["parameter"] == "pm1") & (df["value"] < 0))]
df = df[~((df["parameter"] == "pm1") & (df["value"] > 500))]
df = df[~((df["parameter"] == "pm25") & (df["value"] < 0))]
df = df[~((df["parameter"] == "pm25") & (df["value"] > 999))]
df = df[~((df["parameter"] == "temperature") & (df["value"] < -10))]
df = df[~((df["parameter"] == "temperature") & (df["value"] > 60))]
df = df[~((df["parameter"] == "relativehumidity") & (df["value"] < 0))]
df = df[~((df["parameter"] == "relativehumidity") & (df["value"] > 100))]
print(f"  After outliers   : {len(df):,} rows ({before - len(df)} removed)")

pm1_sensors = df[df["parameter"] == "pm1"][["datetime_utc", "sensor_id"]].rename(
    columns={"sensor_id": "sensor_id_pm1"}
)

df_wide = df.pivot_table(
    index=["datetime_utc", "datetime_local"],
    columns="parameter",
    values="value",
    aggfunc="mean"
).reset_index()

df_wide.columns.name = None
print(f"  After pivot      : {len(df_wide):,} rows")

df_wide = df_wide.merge(pm1_sensors, on="datetime_utc", how="left")

df_wide["city"] = "Kathmandu"
df_wide["latitude"] = 27.7089
df_wide["longitude"] = 85.3397
df_wide["location_name"] = "Gaushala_Chwok_Kathmandu"

df_wide["date"] = df_wide["datetime_utc"].dt.strftime("%Y-%m-%d")

param_cols = ["pm1", "pm25", "temperature", "relativehumidity"]
df_wide = df_wide.dropna(subset=param_cols, how="all")

df_wide = df_wide.rename(columns={
    "pm1": "Parameter(pm1)",
    "pm25": "Parameter(pm2.5)",
    "temperature": "temperature",
    "relativehumidity": "relativehumidity",
})

final_cols = [
    "sensor_id_pm1",
    "datetime_utc",
    "datetime_local",
    "Parameter(pm1)",
    "Parameter(pm2.5)",
    "temperature",
    "relativehumidity",
    "city",
    "latitude",
    "longitude",
    "date",
]

final_cols = [c for c in final_cols if c in df_wide.columns]
df_wide = df_wide[final_cols]
df_wide = df_wide.rename(columns={"sensor_id_pm1": "sensor_id"})

df_wide = df_wide.sort_values("datetime_utc").reset_index(drop=True)

df_wide.to_csv(OUTPUT, index=False)

print(f"\n  SAVED    : {OUTPUT}")
print(f"  Rows     : {len(df_wide):,}")
print(f"  Columns  : {df_wide.columns.tolist()}")

print(f"\n  Stats:")
for col in ["Parameter(pm1)", "Parameter(pm2.5)", "temperature", "relativehumidity"]:
    if col in df_wide.columns:
        print(
            f"  {col:25} mean={df_wide[col].mean():.2f}  "
            f"min={df_wide[col].min():.2f}  max={df_wide[col].max():.2f}  "
            f"nulls={df_wide[col].isnull().sum()}"
        ) 